In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
RAW = Path('../data/raw')

In [ ]:
results     = pd.read_csv(RAW / 'results.csv',      parse_dates=['date'])
rankings    = pd.read_csv(RAW / 'fifa_ranking.csv', parse_dates=['rank_date'])
goalscorers = pd.read_csv(RAW / 'goalscorers.csv')
shootouts   = pd.read_csv(RAW / 'shootouts.csv')

for name, df in [('results', results), ('rankings', rankings), ('goalscorers', goalscorers), ('shootouts', shootouts)]:
    print(f'{name}: {df.shape}')

In [ ]:
results.head()

In [ ]:
results.isnull().sum()

In [ ]:
print(results['date'].min(), '->', results['date'].max())
print('total matches:', len(results))

In [ ]:
# outcome breakdown for matches since the 2022 WC
recent = results[results['date'] >= '2022-12-19'].copy()

recent['result'] = np.where(
    recent['home_score'] > recent['away_score'], 'Home Win',
    np.where(recent['home_score'] == recent['away_score'], 'Draw', 'Away Win')
)

counts = recent['result'].value_counts()
print(counts)
print(f'\nbaseline accuracy: {counts.max()/len(recent):.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Outcomes since 2022 WC')

recent['total_goals'] = recent['home_score'] + recent['away_score']
axes[1].hist(recent['total_goals'], bins=range(0, 12), edgecolor='white', color='steelblue')
axes[1].set_xlabel('total goals')
axes[1].set_ylabel('matches')
axes[1].set_title('Goals per game')

plt.tight_layout()
plt.show()

In [ ]:
rankings.head()

In [ ]:
# latest rankings
latest = rankings[rankings['rank_date'] == rankings['rank_date'].max()]
latest.sort_values('rank')[['rank', 'country_full', 'total_points']].head(20)

In [ ]:
# check team name mismatches between the two datasets
results_teams  = set(results['home_team']) | set(results['away_team'])
rankings_teams = set(rankings['country_full'])

missing = sorted(results_teams - rankings_teams)
print(f'{len(missing)} teams in results not found in rankings:')
for t in missing[:20]:
    print(' ', t)

In [ ]:
# all-time WC goals by nation
wc = results[results['tournament'] == 'FIFA World Cup']
goals = (wc.groupby('home_team')['home_score'].sum()
          .add(wc.groupby('away_team')['away_score'].sum(), fill_value=0)
          .sort_values(ascending=False)
          .head(15))

goals.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.xlabel('WC goals scored')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()